# Where the other local measures sit

From `FINDINGS.md`:

> The screen is `output/measure_screening.csv`. A measure besides the four above was left unmapped when the citywide yes share was more than 5 points from 50 and the east-west correlation was under 0.5. Supervisor-district spreads of 18.2 to 28.5 points are ordinary here, so spread alone did not earn a map.
>
> | Measure | Yes share | East-west r | In one line |
> | --- | ---: | ---: | --- |
> | A, charter changes to departments and commissions | 68.2% | −0.06 | Passed, little east-west tilt |
> | C, contributions to the housing fund | 81.7% | 0.46 | Passed in a landslide, some eastward lean |
> | E, city administrator and contracting | 64.0% | −0.03 | Passed, little east-west tilt |
> | F, executive branch management | 59.7% | −0.05 | Passed, little east-west tilt |
> | H, parcel tax for Muni | 74.3% | 0.37 | Passed, some eastward lean |
> | J, foreclosure exemption on the transfer tax | 80.9% | 0.35 | Passed in a landslide, some eastward lean |
> | RTM, regional transit measure | 79.0% | 0.42 | Passed in a landslide, some eastward lean |

Every number below is computed from the files in `source/`. Run the setup cell first, then the rest in order (Runtime → Run all). The last cell lists each number the memo section quotes next to the value computed here, and stops with an error if any differ.

In [1]:
# Run this cell first. It finds the sf-2026-drive folder (on Google Drive in Colab, or on
# this computer) and loads the shared code in notebooks/sf2026.py.
import subprocess
import sys
from pathlib import Path

try:
    import geopandas  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "geopandas"], check=True)

FOLDER_NAME = "sf-2026-drive"


def is_root(path: Path) -> bool:
    return (path / "source" / "precinct_results.csv").exists() and (path / "notebooks" / "sf2026.py").exists()


def find_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if is_root(candidate):
            return candidate
    try:
        from google.colab import drive
    except ImportError:
        raise FileNotFoundError("Open this notebook from inside the sf-2026-drive folder.")
    drive.mount("/content/drive")
    bases = [Path("/content/drive/MyDrive"), Path("/content/drive/Shareddrives")]
    for base in bases:
        if is_root(base / FOLDER_NAME):
            return base / FOLDER_NAME
    for base in bases:
        if base.exists():
            for helper in base.rglob("sf2026.py"):
                if is_root(helper.parent.parent):
                    return helper.parent.parent
    raise FileNotFoundError(
        f"Could not find the {FOLDER_NAME} folder on Drive. If you renamed it, set FOLDER_NAME above."
    )


ROOT = find_root()
sys.path.insert(0, str(ROOT / "notebooks"))
import pandas as pd
import sf2026 as sf

pd.set_option("display.float_format", lambda value: f"{value:,.2f}")
data = sf.load(ROOT)
check = sf.MemoCheck()
print(f"Using the {ROOT.name} folder: {len(data)} precincts")

Using the sf-2026-drive folder: 514 precincts


## The screen

A local measure gets a map when its citywide yes share is within 5 points of 50, or
when its precinct yes share has an east-west correlation of 0.5 or more (either sign).
The correlation uses precincts with at least 50 votes on the measure.

In [2]:
titles = sf.read_json(ROOT, "measure_titles.json")
screen = sf.screen_measures(data, sf.local_measures(data)).set_index("contest")
screen["title"] = screen.index.map(titles)
screen["mapped"] = ((screen["yes_pct"] - 50).abs() <= 5) | (screen["east_west_r"].abs() >= 0.5)
mapped = sorted(screen.index[screen["mapped"]])
check.fact("Measures that earn a map", "B, D, G, I", mapped,
           mapped == ["Measure B", "Measure D", "Measure G", "Measure I"])
check.number("Smallest supervisor-district spread, points", 18.2, screen["district_spread_points"].min())
check.number("Largest supervisor-district spread, points", 28.5, screen["district_spread_points"].max())
screen[["title", "yes_pct", "east_west_r", "district_spread_points", "mapped"]]

,title,yes_pct,east_west_r,district_spread_points,mapped
contest,,,,,
Measure A,Charter Changes Affecting Various City Departm...,68.18,-0.06,23.03,False
Measure B,Establishing A Municipal Finance Corporation A...,52.31,0.35,23.06,True
Measure C,Contributions To The Housing Fund,81.69,0.46,19.60,False
Measure D,Changes To Ballot Measure Process,54.46,0.01,23.03,True
Measure E,City Administrator’s Authority And Changes To ...,63.98,-0.03,24.09,False
Measure F,Changes To Executive Branch Management,59.69,-0.05,25.36,False
Measure G,Allowing Private Vehicles On The Great Highway...,34.44,-0.52,28.53,True
Measure H,Parcel Tax To Fund Public Muni Operations,74.27,0.37,24.66,False
Measure I,Changes To Real Property Transfer Tax,47.94,0.32,21.78,True


In [3]:
table = {"A": (68.2, -0.06), "C": (81.7, 0.46), "E": (64.0, -0.03), "F": (59.7, -0.05),
         "H": (74.3, 0.37), "J": (80.9, 0.35), "RTM": (79.0, 0.42)}
for letter, (yes_pct, east_west) in table.items():
    row = screen.loc[f"Measure {letter}"]
    check.number(f"Measure {letter} yes share, percent", yes_pct, row["yes_pct"])
    check.number(f"Measure {letter} east-west r", east_west, row["east_west_r"], 2)
    check.fact(f"Measure {letter} passed", "passed", f"{row['yes_pct']:.1f}% yes", row["yes_pct"] > 50)

## Numbers the memo quotes

In [4]:
check.report();

,claim,memo,computed,matches
0,Measures that earn a map,"B, D, G, I","['Measure B', 'Measure D', 'Measure G', 'Measure I']",yes
1,"Smallest supervisor-district spread, points",18.2,18.2,yes
2,"Largest supervisor-district spread, points",28.5,28.5,yes
3,"Measure A yes share, percent",68.2,68.2,yes
4,Measure A east-west r,-0.06,-0.06,yes
5,Measure A passed,passed,68.2% yes,yes
6,"Measure C yes share, percent",81.7,81.7,yes
7,Measure C east-west r,0.46,0.46,yes
8,Measure C passed,passed,81.7% yes,yes
9,"Measure E yes share, percent",64.0,64.0,yes


All 24 numbers checked match the memo.
